In [1]:
from deepface import DeepFace
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import numpy as np
import json
import time
import cv2
import csv
import os

In [2]:

def log_access(person_name,is_real,authorized,face_image=None,log_path="access_log.csv",save_faces=True):

    os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)

    file_exists = os.path.exists(log_path)

    with open(log_path, mode="a", newline="") as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(["Timestamp", "Person", "Spoofing", "Authorized", "Face_Image"])

        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

        spoof_status = "Real" if is_real else "Fake"
        auth_status = "Yes" if authorized else "No"
        image_filename = ""

        if save_faces and face_image is not None:
            folder = "Authorized" if authorized else "Unauthorized"
            os.makedirs(folder, exist_ok=True)

            clean_name = person_name.replace(" ", "_")
            image_filename = os.path.join(folder, f"{timestamp}_{clean_name}.jpg")

            cv2.imwrite(image_filename, face_image)

        writer.writerow([timestamp, person_name, spoof_status, auth_status, image_filename])

    print(f"[LOG] {person_name} | Spoofing: {spoof_status} | Authorized: {auth_status}")


In [3]:

def load_model():
    face_datector = YOLO("yolov11n-face.pt")
    face_regonizer = DeepFace
    return face_datector, face_regonizer


In [4]:

def Init_Camera():
    cap = cv2.VideoCapture(0)
    print("Camera is starting...")
    if not cap.isOpened():
        print("Cannot open camera")
        exit()
    return cap


In [5]:

def get_picture(camera):
    print("Getting frame from camera...")
    ret, frame = camera.read() # ret = Return value, frame = the captured image
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        raise SystemExit("Can't receive frame (stream end?). Exiting ...")
    return frame

In [6]:
def close_Camera(camera):
    camera.release()
    cv2.destroyAllWindows()



In [7]:
def detect_face(face_datector, frame):
    results = face_datector(frame)
    boxes = results[0].boxes.xyxy.cpu().numpy()  # Extract bounding box coordinates
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)  # Convert coordinates to integers
        cropped_face = frame[y1-75:y2+75, x1-75:x2+75]  # Crop the face from the frame
        Num = i + 1
    return cropped_face, Num


In [8]:
def recognize_face(input_, database):
    try:
        result = DeepFace.find(input_, database,enforce_detection=False)
        print(str(result))
        
        df = pd.DataFrame(result[0])
        record = json.loads(df.to_json(orient="records"))

        with open("match.json", "w") as f:
            for i in record:
                try:
                    if i['confidence'] >= 65:
                        f.write(json.dumps(i["identity"].strip("faceDB/"), indent=2))
                        f.write("\n")
                except Exception as e:
                    print("Error writing identity: ", str(e))
                    return False, None
            print("JSON File Saved")
        return True, record[0]
    
    except Exception as e:
        print("Some errors occurred: ", str(e))
        return False, None


In [9]:
face_datector, face_regonizer= load_model()
print("Models loaded.")


Models loaded.


In [39]:
start_time = time.time()
print("Initializing camera...")
camera = Init_Camera()
try:
        image = get_picture(camera)
        # image = cv2.imread("out.jpg")  # For testing unathorized user
        if image is not None:
            cv2.imwrite("Raw.jpg", image)
            print(f"Raw saved. Image shape: {image.shape}")
            Face, FaceNum = detect_face(face_datector, image)
            print("Face found",FaceNum)
            if FaceNum > 1:
                print("Multiple faces detected. Please ensure only one face is in the frame.")
                raise SystemExit("Exiting due to multiple faces.")
            cv2.imwrite("cropped_face.jpg", Face)
            print(f"Cropped_face saved. Cropped shape: {Face.shape}")
            Spoof_check = DeepFace.extract_faces(Face, anti_spoofing=True)
            Real = Spoof_check[0]["is_real"]
            print(type(Real))
            print(f"Spoofing check result: {Real}")
            if Real is False:
                print("Spoofing detected! Access denied.")
            found, who = recognize_face(Face, "faceDB")   
        #     identity = who["identity"] not work because who may be None
        #     who_name = os.path.splitext(os.path.basename(identity))[0] #basename = Remove path(Ongsa1.jpg), splitext = remove extension(Ongsa1 - .jpg)    
        if(found and Real is True):
                identity = who["identity"]
                who_name = os.path.splitext(os.path.basename(identity))[0] #basename = Remove path(Ongsa1.jpg), splitext = remove extension(Ongsa1 - .jpg)
                print(f"Found face is database! Welcome back, {who_name}😊🙏🥀")
                log_access(person_name=who_name, is_real=Real, authorized=True, face_image=Face)
        elif(found and Real is False):
                print("Get the hell out of here, you are not authorized!😠👊🚪")
                identity = who["identity"]
                who_name = os.path.splitext(os.path.basename(identity))[0] #basename = Remove path(Ongsa1.jpg), splitext = remove extension(Ongsa1 - .jpg)                
                log_access(person_name=who_name, is_real=Real, authorized=False, face_image=Face)
        else:
                print("Get the hell out of here, you are not authorized!😠👊🚪")
                log_access(person_name="Unknown", is_real=Real, authorized=False, face_image=Face)

except SystemExit as e:
        print(e)       
finally:
        print("Releasing camera...")
        close_Camera(camera)
end_time = time.time()
print(f"Total time taken: {end_time - start_time} seconds")

Initializing camera...
Camera is starting...
Getting frame from camera...
Raw saved. Image shape: (480, 640, 3)

0: 480x640 1 face, 15.4ms
Speed: 1.8ms preprocess, 15.4ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
Face found 1
Cropped_face saved. Cropped shape: (221, 201, 3)
<class 'bool'>
Spoofing check result: False
Spoofing detected! Access denied.
25-11-05 23:28:37 - Searching [[[ 99  97 107]
  [ 81  79  89]
  [ 58  54  65]
  ...
  [ 66  56  63]
  [ 65  55  62]
  [ 67  56  63]]

 [[ 97  95 105]
  [ 72  69  79]
  [ 62  58  67]
  ...
  [ 66  57  64]
  [ 66  57  64]
  [ 64  55  62]]

 [[ 98  96 106]
  [ 72  69  79]
  [ 63  59  68]
  ...
  [ 65  57  64]
  [ 66  58  65]
  [ 65  57  64]]

 ...

 [[ 87  86  99]
  [ 99  98 111]
  [198 198 208]
  ...
  [166 161 173]
  [114 107 124]
  [110 105 120]]

 [[ 87  87  98]
  [ 98  98 109]
  [198 198 208]
  ...
  [156 152 164]
  [114 108 125]
  [114 109 125]]

 [[ 88  89 100]
  [ 97  98 109]
  [196 198 206]
  ...
  [146 142 154